# CS 691 · Module 16 · Capstone: auditing a hospital teaching-room handoff

Build and test an evidence-backed review of training-room equipment handoffs, then explain when an administrative reviewer needs to inspect the records.

**Before you begin.** You have built rule traces, checked evidence, compared event times and evaluated held-out decisions. You will combine these skills in one new setting.

A staff note and a scan both concern ticket T40. The note records a check at minute 6 and the scan records a handoff at minute 9. Assuming the note is accepted as evidence, which event came first and by how much?

<details><summary>Reveal the answer</summary>

The recorded check came first by three minutes. Both rows refer to T40, so the time comparison concerns one handoff.

</details>

## Translate the clerk’s question into a rule

A fictional hospital teaching room lends projectors to staff for workshops. Its clerk asks whether the paperwork records a label check before each handoff. This is an administrative record-review task. A teaching-room ticket identifies one handoff, just as a request ID identified Ava’s camera loan.

Ticket T17 concerns projector P7. K8 is a note attached to a recorded label check. H4 is a scanned handoff record. Keep evidence from the same ticket. Each row below gives the evidence ID, ticket and source type.

**Predict before running the cell.** A check note for T18 also mentions projector P7. May the clerk use it to clear T17?

In [ ]:
records = [("K8", "T17", "note"), ("H4", "T17", "scan"),
           ("K9", "T18", "note")]
evidence = [key for key, ticket, kind in records if ticket == "T17"]
print(", ".join(evidence))

<details><summary>Reveal output and explanation</summary>

```text
K8, H4
```

The projector can be borrowed several times. T17 needs a check record tied to its own handoff. The shared item name cannot establish that link.

</details>

## Connect a learned signal to an evidence record

A note classifier reads a staff note and scores whether it reports a completed label check. The practice data supply its scores so everyone can audit the same inputs. Under the exercise policy, a score of at least 0.80 admits the note as candidate evidence. A lower score sends the case for review.

K8 reads “Projector P7 label checked” and receives 0.92. K12 reads “Check label when available” and receives 0.55. The source wording remains available to the clerk even when a score passes the cutoff.

**Predict before running the cell.** If a note only promises a future check but receives 0.93, what can the rule establish?

In [ ]:
notes = [("K8", 0.92), ("K12", 0.55)]
for source, score in notes:
    result = "use as candidate" if score >= 0.80 else "review note"
    print(source, result)

<details><summary>Reveal output and explanation</summary>

```text
K8 use as candidate
K12 review note
```

It can establish that the classifier admitted that note. The wording does not document a completed check. A clerk who examines the source can catch the classification error.

</details>

## Return a finding the clerk can inspect

Use one candidate note and one handoff scan for the same ticket. Compare their event minutes. Label the records “in order” or “reversed.” Use “needs review” for missing evidence, a low score or tied times. These labels summarize the records for a human reviewer.

For T17, K8 reports a check at minute 12 and H4 records handoff at 18. Print the ticket, finding and source IDs so the clerk can open the records. This case has a qualifying note and distinct times.

**Predict before running the cell.** What would you write for the clerk, and what could the clerk check?

In [ ]:
ticket = "T17"
note_id, score, check = "K8", 0.92, 12
scan_id, handoff = "H4", 18
if None in (note_id, score, check, scan_id, handoff) or score < 0.80 or check == handoff:
    finding = "needs review"
elif check < handoff:
    finding = "in order"
else:
    finding = "reversed"
print(ticket, finding, note_id, scan_id)

<details><summary>Reveal output and explanation</summary>

```text
T17 in order K8 H4
```

“For T17, candidate note K8 reports a check at minute 12. Scan H4 records a handoff at minute 18. The recorded order meets the teaching-room rule.” The clerk can verify the note wording, times and ticket IDs.

</details>

## Run the audit on three tickets

Your capstone is a small program plus a short audit. First predict the findings for the three tickets below. Then run the code, trace the source IDs and add the two tests in the exercises: one for the cutoff boundary and one for tied times. Treat None as missing evidence. The compact table assumes the desk has already joined each ticket to its own source rows.

Rows give a ticket, note ID, classifier score, check minute, scan ID and handoff minute. T17 and T18 have both sources. T19’s note is uncertain and its scan is missing. Each ticket has at most one note and one scan in this exercise.

**Predict before running the cell.** What would make this submission convincing beyond matching the three printed labels?

In [ ]:
cases = [("T17", "K8", .92, 12, "H4", 18),
         ("T18", "K9", .90, 20, "H5", 17),
         ("T19", "K12", .55, 11, None, None)]
for ticket, note, score, check, scan, handoff in cases:
    if None in (note, score, check, scan, handoff) or score < .80:
        result = "needs review"
    elif check == handoff:
        result = "needs review"
    else:
        result = "in order" if check < handoff else "reversed"
    print(ticket, result, note, scan)

<details><summary>Reveal output and explanation</summary>

```text
T17 in order K8 H4
T18 reversed K9 H5
T19 needs review K12 None
```

For each decision, explain what the score test and time test contributed, cite the available records and name any missing evidence. Test a new case such as a high score with tied timestamps. Check that the clerk can see the uncertainty.

</details>

## Trace ticket T17 back to both sources

The first condition checks the required records, the note score and a possible tie. If these checks pass, comparing the minutes decides between in order and reversed. The final line prints the ticket, finding and source IDs together so a clerk can check the original entries.

Replace this line:
scan_id, handoff = "H4", 18
with:
scan_id, handoff = "H4", 12
What will print?

In [ ]:
ticket = "T17"
note_id, score, check = "K8", 0.92, 12
scan_id, handoff = "H4", 18
if None in (note_id, score, check, scan_id, handoff) or score < 0.80 or check == handoff:
    finding = "needs review"
elif check < handoff:
    finding = "in order"
else:
    finding = "reversed"
print(ticket, finding, note_id, scan_id)

<details><summary>Worked answer after your change</summary>

T17 needs review K8 H4

Both observations now carry minute 12. The recorded times do not establish which event came first, and the report retains both source IDs for review.

</details>

## Exercise 1

Build the capstone review for T17, T18 and T19. Report each status with its source IDs and explain the tests in everyday language.

<details><summary>Hint</summary>

Retrieve within a ticket, apply the inclusive 0.80 cutoff, check that both records exist, then compare times. Look for every reason a case needs review.

</details>

<details><summary>Worked answer</summary>

T17 is in order on the records: K8 scores 0.92 and reports minute 12 before H4 at 18. T18 is reversed: K9 scores 0.90 but reports 20 after H5 at 17. T19 needs review: K12 scores 0.55 and there is no scan. Its evidence cannot establish the order.

</details>

## Exercise 2

Add a ticket that isolates the cutoff boundary and one that isolates equal times. Predict both outputs before running your program.

<details><summary>Hint</summary>

Keep all other facts ordinary and complete. Change one feature per case.

</details>

<details><summary>Worked answer</summary>

Try T20, K13, 0.80, 7, H6, 8. The inclusive cutoff admits K13 and the result is in order. Try T21, K14, 0.95, 9, H7, 9. The note passes, but minute-level times tie, so the result is needs review.

</details>

## Exercise 3

A clerk independently labels four new tickets: in order, reversed, needs review, needs review. The system gives in order, in order, needs review, in order. Evaluate it and propose one diagnostic step.

<details><summary>Hint</summary>

Line up labels by ticket. Identify a reversal it missed and a case it cleared with insufficient evidence.

</details>

<details><summary>Worked answer</summary>

Two of four labels agree. The second ticket is a missed reversal. The fourth is an unsupported clearance. Inspect source IDs, wording and timestamps for each, replay the cutoff and order tests, and report which step first diverged from the clerk’s evidence. For a new version, keep fresh cases for independent evaluation.

</details>

Return to the course overview and trace one finding from its learned input through its rule and supporting records. Your capstone is complete when another reader can reproduce the finding, identify its assumptions and tell when to ask for review.

A second reviewer reproduces an “in order” result, then learns that the qualifying note’s minute was its filing time, not its event time. What should appear in the audit now?

<details><summary>Reveal the answer</summary>

Record the discovered assumption and mark the order for review until the actual check time is available. Reproducing the computation shows how the first label arose, but the available times no longer establish the sequence.

</details>